# EARS Geothermal — Factorial Results Analysis

Analyze and visualize solved scenario networks of the main factorial
(`MODELING_PLAN.md` §6): **year (2030/40/50) × hydro (normal/dry) × demand (GEGIS/HEG) × geothermal (geo/nogeo)**.

**How to use:** paste the path(s) of the solved `.nc` file(s) into the `PASTE`
list in the first code cell and run the notebook top-to-bottom. Any path works
(absolute or relative). Factorial coordinates (year/hydro/demand/geo) are
inferred from the filename where possible — they're only needed for the
value-of-geothermal pairing, everything else works regardless.

Files named `<year>_<hydro>_<demand>_<geo>.nc` in `scenarios/` (e.g.
`2050_dry_HEG_geo.nc`) are additionally picked up automatically.

**Two rules to never forget when reading numbers:**
1. **Snapshot weighting** — energy = dispatch × `snapshot_weightings.objective`. Every function here does this; don't sum raw time series.
2. **Resolution caveat** — 168H (weekly) runs erase the diurnal cycle: solar looks firm, storage looks useless (validated 2026-07-12: same cell at 168H vs 4H → battery 0 vs 267 GWh, cost +73%). Only ≥4H runs are thesis-reportable (§3e).

In [ ]:
import glob, os, re, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pypsa

# silence PyPSA's per-network import chatter (one WARNING+INFO block per file,
# ~19 files = a wall of text). Note: if you see a "v0.30.3 while current v1.2.x"
# warning, the notebook is on the WRONG kernel -- switch to "Python (pypsa-earth)"
# (the 0.30.3 env that solved the networks) rather than just muting it.
logging.getLogger("pypsa").setLevel(logging.ERROR)

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# consistent carrier colors across all plots
COLORS = {
    "hydro": "#2166ac", "ror": "#67a9cf", "geothermal": "#b2182b",
    "solar": "#fdb863", "onwind": "#5aae61", "oil": "#4d4d4d",
    "battery": "#9970ab", "H2": "#c2a5cf", "load": "#000000",
    "waste": "#8c510a", "biomass": "#8c510a", "OCGT": "#878787", "PHS": "#51dbcc",
}
CARRIER_ORDER = ["hydro", "ror", "geothermal", "solar", "onwind", "oil", "OCGT", "waste", "load"]

# ════════════════════════════════════════════════════════════════════════════
# PASTE the path(s) of the solved .nc file(s) you want to analyze here.
# Absolute or relative paths both work; raw strings (r"...") save you from
# escaping Windows backslashes. Everything below analyzes whatever is listed.
#
# NOTE: for the Zuffi-vs-JICA cost comparison, skip this -- that section (further
# down) loads its own matched pairs automatically from scenarios/ + scenarios/sensitivity/.
# ════════════════════════════════════════════════════════════════════════════
PASTE = [
]

# Factorial cells named <year>_<hydro>_<demand>_<geo>.nc in scenarios/ are ALSO
# picked up automatically (set AUTO_DISCOVER = False to analyze only PASTE).
AUTO_DISCOVER = True
PATTERN = re.compile(r"(?P<year>20[345]0)_(?P<hydro>normal|dry)_(?P<demand>GEGIS|HEG)_(?P<geo>geo|nogeo)\.nc$")

def parse_meta(path):
    """Infer factorial coordinates from the filename where possible.
    Also tags lcoe: 'jica' if the filename mentions it, else 'zuffi' (default)."""
    name = os.path.basename(path)
    m = PATTERN.search(name)
    if m:
        d = m.groupdict()
    else:
        low = name.lower()
        d = dict(
            year=next((y for y in ("2030", "2040", "2050") if y in low), "?"),
            hydro="dry" if "dry" in low else ("normal" if "normal" in low else "?"),
            demand="HEG" if "heg" in low else ("GEGIS" if "gegis" in low else "?"),
            geo="nogeo" if "nogeo" in low or "frozen" in low else ("geo" if "geo" in low else "?"),
        )
    d["lcoe"] = "jica" if "jica" in name.lower() else "zuffi"
    return d

RUNS = {}
for p in PASTE:
    if not os.path.isfile(p):
        print(f"  !! not found, skipped: {p}")
        continue
    label = os.path.splitext(os.path.basename(p))[0]
    RUNS[label] = dict(path=p, **parse_meta(p))
if AUTO_DISCOVER:
    for p in sorted(glob.glob(os.path.join("scenarios", "*.nc"))):
        m = PATTERN.search(os.path.basename(p))
        if m:
            d = m.groupdict()
            label = f"{d['year']}·{d['hydro']}·{d['demand']}·{d['geo']}"
            RUNS.setdefault(label, dict(path=p, lcoe="zuffi", **d))

print(f"{len(RUNS)} runs registered:")
for k, v in RUNS.items():
    print(f"  {k:40s} {v['path']}")
if not RUNS:
    print("  -> paste at least one .nc path into PASTE above and re-run this cell "
          "(or see the Zuffi-vs-JICA section further down, which needs no pasting)")

## Metric extraction (weighted, cached)

In [ ]:
_cache = {}

def load_net(path):
    if path not in _cache:
        _cache[path] = pypsa.Network(path)
    return _cache[path]

def energy_by_carrier(n):
    """Annual generation per carrier in TWh (generators + storage-unit discharge), weighted."""
    w = n.snapshot_weightings.objective
    gen = n.generators_t.p.mul(w, axis=0).sum().groupby(n.generators.carrier).sum()
    sug = n.storage_units_t.p.clip(lower=0).mul(w, axis=0).sum().groupby(n.storage_units.carrier).sum()
    return pd.concat([gen, sug]).groupby(level=0).sum() / 1e6

def metrics(path):
    n = load_net(path)
    w = n.snapshot_weightings.objective
    e = energy_by_carrier(n)
    dem = n.loads_t.p_set.mul(w, axis=0).sum().sum() / 1e6
    geo = n.generators[n.generators.carrier == "geothermal"]
    ext = n.generators[n.generators.p_nom_extendable & ~n.generators.carrier.isin(["load"])]
    built = (ext.p_nom_opt - ext.p_nom).groupby(ext.carrier).sum()
    stores = n.stores.groupby(n.stores.carrier).e_nom_opt.sum() if len(n.stores) else pd.Series(dtype=float)
    geo_gen = e.get("geothermal", 0.0)
    geo_mw = geo.p_nom_opt.sum()
    return dict(
        demand_TWh=dem,
        cost_BEUR=n.objective / 1e9,
        unserved_GWh=e.get("load", 0.0) * 1e3,
        unserved_pct=e.get("load", 0.0) / dem * 100 if dem else np.nan,
        geo_built_MW=geo_mw,
        geo_gen_TWh=geo_gen,
        geo_share_pct=geo_gen / dem * 100 if dem else np.nan,
        geo_flh=geo_gen * 1e6 / geo_mw if geo_mw > 0 else np.nan,
        solar_built_MW=built.get("solar", 0.0),
        wind_built_MW=built.get("onwind", 0.0),
        battery_GWh=stores.get("battery", 0.0) / 1e3,
        H2_GWh=stores.get("H2", 0.0) / 1e3,
        hydro_gen_TWh=e.get("hydro", 0.0) + e.get("ror", 0.0),
    )

summary = pd.DataFrame({label: metrics(v["path"]) for label, v in RUNS.items()}).T
meta = pd.DataFrame({label: {k: v[k] for k in ("year", "hydro", "demand", "geo")} for label, v in RUNS.items()}).T
summary = meta.join(summary)
summary

## Generation mix per scenario (stacked)

In [ ]:
# Split GEGIS vs HEG into separate panels: HEG demand (112-289 TWh) is ~7-18x
# GEGIS (16-34 TWh), so on a shared axis the GEGIS bars are invisible. Each
# panel gets its own y-scale.
mix = pd.DataFrame({label: energy_by_carrier(load_net(v["path"])) for label, v in RUNS.items()}).T.fillna(0)
cols = [c for c in CARRIER_ORDER if c in mix.columns] + [c for c in mix.columns if c not in CARRIER_ORDER]
mix = mix[cols]
demand_of = {label: v["demand"] for label, v in RUNS.items()}

panels = [d for d in ("GEGIS", "HEG") if any(demand_of.get(l) == d for l in mix.index)]
fig, axes = plt.subplots(1, len(panels), figsize=(7.5 * len(panels), 5), squeeze=False)
axes = axes[0]
for ax, dem in zip(axes, panels):
    labels = [l for l in mix.index if demand_of.get(l) == dem]
    sub = mix.loc[labels]
    sub.plot(kind="bar", stacked=True, ax=ax, legend=False,
             color=[COLORS.get(c, "#cccccc") for c in sub.columns])
    ax.set_title(f"{dem} demand"); ax.set_ylabel("TWh/yr")
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=45)
    for lab in ax.get_xticklabels():
        lab.set_ha("right")
axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.suptitle("Annual generation by carrier — split by demand scenario", y=1.02)
plt.tight_layout()

## New capacity built per scenario

In [ ]:
# New capacity built, split GEGIS | HEG (same reason as the generation panel).
cap = summary[["geo_built_MW", "solar_built_MW", "wind_built_MW"]].astype(float)
cap.columns = ["geothermal", "solar", "onwind"]
demand_of = {label: v["demand"] for label, v in RUNS.items()}
panels = [d for d in ("GEGIS", "HEG") if any(demand_of.get(l) == d for l in cap.index)]

fig, axes = plt.subplots(1, len(panels), figsize=(7.5 * len(panels), 4.5), squeeze=False)
axes = axes[0]
for ax, dem in zip(axes, panels):
    labels = [l for l in cap.index if demand_of.get(l) == dem]
    cap.loc[labels].plot(kind="bar", ax=ax, legend=(ax is axes[-1]),
                         color=[COLORS[c] for c in cap.columns])
    ax.set_title(f"{dem} demand"); ax.set_ylabel("MW built")
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=45)
    for lab in ax.get_xticklabels():
        lab.set_ha("right")
fig.suptitle("New capacity by technology (p_nom_opt − p_nom) — split by demand", y=1.02)
plt.tight_layout()

# storage on its own figure (GWh; also split by demand)
st = summary[["battery_GWh", "H2_GWh"]].astype(float)
if st.values.sum() > 0.1:
    fig, axes = plt.subplots(1, len(panels), figsize=(7.5 * len(panels), 3), squeeze=False)
    axes = axes[0]
    for ax, dem in zip(axes, panels):
        labels = [l for l in st.index if demand_of.get(l) == dem]
        st.loc[labels].plot(kind="bar", ax=ax, legend=(ax is axes[-1]),
                            color=[COLORS["battery"], COLORS["H2"]])
        ax.set_title(f"{dem} demand"); ax.set_ylabel("GWh built")
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", rotation=45)
        for lab in ax.get_xticklabels():
            lab.set_ha("right")
    fig.suptitle("Storage energy capacity built — split by demand", y=1.02)
    plt.tight_layout()

## Value of geothermal — the §8 delta
`Value(geo | cell) = X(nogeo) − X(geo)` for each dry cell that has both twins.

In [ ]:
pairs = []
for label, v in RUNS.items():
    if v["geo"] != "geo":
        continue
    # find the nogeo twin by matching factorial coordinates (works for pasted
    # files with arbitrary names, as long as year/hydro/demand were inferred)
    twin = next((t for t, u in RUNS.items()
                 if u["geo"] == "nogeo"
                 and u["year"] == v["year"] and u["hydro"] == v["hydro"]
                 and u["demand"] == v["demand"] and "?" not in (v["year"], v["hydro"], v["demand"])),
                None)
    if twin is None:
        continue
    a, b = summary.loc[label], summary.loc[twin]
    pairs.append({
        "cell": f"{v['year']}·{v['hydro']}·{v['demand']}",
        "Δcost_BEUR": b.cost_BEUR - a.cost_BEUR,
        "Δunserved_GWh": b.unserved_GWh - a.unserved_GWh,
        "Δsolar_MW": b.solar_built_MW - a.solar_built_MW,
        "Δwind_MW": b.wind_built_MW - a.wind_built_MW,
        "Δbattery_GWh": b.battery_GWh - a.battery_GWh,
        "geo_built_MW": a.geo_built_MW,
        "geo_gen_TWh": a.geo_gen_TWh,
    })
value = pd.DataFrame(pairs).set_index("cell") if pairs else pd.DataFrame()
if len(value):
    display(value)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    value["Δcost_BEUR"].plot(kind="bar", ax=axes[0], color="#b2182b")
    axes[0].set_title("Δ system cost (nogeo − geo)"); axes[0].set_ylabel("B€/yr"); axes[0].grid(axis="y", alpha=0.3)
    value["Δunserved_GWh"].plot(kind="bar", ax=axes[1], color="#4d4d4d")
    axes[1].set_title("Δ unserved energy (nogeo − geo)"); axes[1].set_ylabel("GWh/yr"); axes[1].grid(axis="y", alpha=0.3)
    for ax in axes: ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
else:
    print("No geo/nogeo twin pairs found — needs a geo run AND its nogeo twin "
          "with matching year/hydro/demand (inferred from filenames).")

## Zuffi vs JICA LCOE sensitivity (§7 bias check)

Self-contained -- loads its own matched pairs, ignores `PASTE`/`RUNS` above.
For every (year, hydro, demand) cell where **both** a Zuffi-cost geo run
(`scenarios/*_geo.nc`) and a JICA-cost geo run (`scenarios/sensitivity/*_geo_jica.nc`)
exist, compares geothermal built and what substitutes for it. Zuffi FLASH LCOE
(~$22-71/MWh) is the central case; JICA's own LCOE (~$57-111/MWh, 2 sites with
no JICA figure dropped) is the conservative national-source sensitivity.

In [ ]:
CELL_PATTERN = re.compile(r"(20[345]0)_(normal|dry)_(GEGIS|HEG)_geo")

def find_geo_cells(folder, want_jica):
    """Map (year,hydro,demand) -> path for every *_geo*.nc in folder that does
    (want_jica=True) or doesn't (False) have 'jica' in its filename."""
    out = {}
    for p in sorted(glob.glob(os.path.join(folder, "*_geo*.nc"))):
        name = os.path.basename(p)
        is_jica = "jica" in name.lower()
        if is_jica != want_jica:
            continue
        m = CELL_PATTERN.search(name)
        if m:
            out[(m.group(1), m.group(2), m.group(3))] = p
    return out

zuffi_cells = find_geo_cells("scenarios", want_jica=False)
jica_cells = find_geo_cells(os.path.join("scenarios", "sensitivity"), want_jica=True)
common = sorted(set(zuffi_cells) & set(jica_cells))
print(f"Zuffi geo cells found: {len(zuffi_cells)} | JICA geo cells found: {len(jica_cells)} "
      f"| matched pairs: {len(common)}")
if not common:
    print("No matched pairs -- check scenarios/*_geo.nc and scenarios/sensitivity/*_geo_jica.nc exist.")

rows = []
for (yr, hyd, dem) in common:
    nz, nj = pypsa.Network(zuffi_cells[(yr, hyd, dem)]), pypsa.Network(jica_cells[(yr, hyd, dem)])
    def m(n):
        w = n.snapshot_weightings.objective
        ext = n.generators[n.generators.p_nom_extendable]
        built = (ext.p_nom_opt - ext.p_nom).groupby(ext.carrier).sum()
        geo = n.generators[n.generators.carrier == "geothermal"]
        gen = n.generators_t.p.mul(w, axis=0).sum().groupby(n.generators.carrier).sum() / 1e6
        batt = n.stores[n.stores.carrier == "battery"].e_nom_opt.sum() / 1e3 if len(n.stores) else 0.0
        return dict(cost=n.objective / 1e9, geo_MW=geo.p_nom_opt.sum(), geo_TWh=gen.get("geothermal", 0.0),
                    solar_GW=built.get("solar", 0.0) / 1e3, wind_GW=built.get("onwind", 0.0) / 1e3, batt_GWh=batt)
    z, j = m(nz), m(nj)
    rows.append({
        "cell": f"{yr}·{hyd}·{dem}",
        "geo_MW_zuffi": z["geo_MW"], "geo_MW_jica": j["geo_MW"], "Δgeo_MW": j["geo_MW"] - z["geo_MW"],
        "cost_zuffi_BEUR": z["cost"], "cost_jica_BEUR": j["cost"], "Δcost_BEUR": j["cost"] - z["cost"],
        "Δsolar_GW": j["solar_GW"] - z["solar_GW"], "Δwind_GW": j["wind_GW"] - z["wind_GW"],
        "Δbatt_GWh": j["batt_GWh"] - z["batt_GWh"],
    })

sens = pd.DataFrame(rows).set_index("cell") if rows else pd.DataFrame()
if len(sens):
    display(sens)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sens[["geo_MW_zuffi", "geo_MW_jica"]].plot(kind="bar", ax=axes[0],
                                               color=["#fdb863", "#b2182b"])
    axes[0].set_title("Geothermal built: Zuffi vs JICA cost"); axes[0].set_ylabel("MW")
    axes[0].grid(axis="y", alpha=0.3)
    sens["Δcost_BEUR"].plot(kind="bar", ax=axes[1], color="#4d4d4d")
    axes[1].set_title("Δ system cost (JICA − Zuffi)"); axes[1].set_ylabel("B€/yr")
    axes[1].grid(axis="y", alpha=0.3)
    for ax in axes:
        ax.tick_params(axis="x", rotation=45)
        for lab in ax.get_xticklabels():
            lab.set_ha("right")
    plt.tight_layout()

## Geothermal build-out per site

In [ ]:
RUN = list(RUNS)[-1]   # ← pick the run to inspect
n = load_net(RUNS[RUN]["path"])
geo = n.generators[n.generators.carrier == "geothermal"].copy()
geo["site"] = geo.index.str.replace("geothermal ", "", regex=False)
site = geo.set_index("site")[["p_nom_opt", "p_nom_max"]].sort_values("p_nom_max", ascending=True)
ax = site.plot(kind="barh", figsize=(7, max(4, 0.32 * len(site))), color=["#b2182b", "#f4a582"])
ax.set_xlabel("MW")
ax.set_title(f"Geothermal build vs cap — {RUN}")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

## Dispatch time series (pick a run and a window)

In [ ]:
RUN = list(RUNS)[-1]          # ← run to plot
WEEK_START = "2013-03-04"     # one-week zoom start (full year at 4H = ~2200 steps is
                             # unreadable). Set WEEK_START = None for the whole year.
                             # Ethiopian dry season ≈ Oct–May.

n = load_net(RUNS[RUN]["path"])
if WEEK_START:
    end = (pd.Timestamp(WEEK_START) + pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    window = slice(WEEK_START, end)
else:
    window = slice(None)

gen = n.generators_t.p.T.groupby(n.generators.carrier).sum().T
sto = n.storage_units_t.p.clip(lower=0).T.groupby(n.storage_units.carrier).sum().T
disp = pd.concat([gen, sto], axis=1)
disp = disp.T.groupby(level=0).sum().T.loc[window] / 1e3   # GW
cols = [c for c in CARRIER_ORDER if c in disp.columns] + [c for c in disp.columns if c not in CARRIER_ORDER]
disp = disp[cols].clip(lower=0)
load = n.loads_t.p_set.sum(axis=1).loc[window] / 1e3

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.stackplot(disp.index, disp.T.values, labels=disp.columns,
             colors=[COLORS.get(c, "#cccccc") for c in disp.columns])
ax.plot(load.index, load.values, "k--", lw=1.2, label="demand")
ax.set_ylabel("GW")
span = "full year" if WEEK_START is None else f"week of {WEEK_START}"
ax.set_title(f"Dispatch — {RUN}  ({span})")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.margins(x=0)
plt.tight_layout()

## Seasonal profile — monthly energy by carrier (the dry-season story)

In [ ]:
RUN = list(RUNS)[-1]         # ← run to plot
n = load_net(RUNS[RUN]["path"])
w = n.snapshot_weightings.objective
gen = n.generators_t.p.mul(w, axis=0).T.groupby(n.generators.carrier).sum().T
sto = n.storage_units_t.p.clip(lower=0).mul(w, axis=0).T.groupby(n.storage_units.carrier).sum().T
monthly = pd.concat([gen, sto], axis=1)
monthly = monthly.T.groupby(level=0).sum().T
monthly = monthly.groupby(monthly.index.month).sum() / 1e3   # GWh
cols = [c for c in CARRIER_ORDER if c in monthly.columns] + [c for c in monthly.columns if c not in CARRIER_ORDER]
ax = monthly[cols].plot(kind="bar", stacked=True, figsize=(10, 4.5),
                        color=[COLORS.get(c, "#cccccc") for c in cols])
ax.set_xlabel("month"); ax.set_ylabel("GWh")
ax.set_title(f"Monthly generation — {RUN}  (Ethiopian dry season ≈ Oct–May)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

## Network map — paper style (Parzen et al. 2023, Fig. 11)

Capacity pies per node; **light grey lines = existing transmission**, **dark rose = newly optimized** capacity (both width-scaled). Set `RUN`, tweak `PIE_SCALE` if pies crowd the map (large-demand scenarios need smaller values).

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature

RUN = list(RUNS)[-1]     # ← run to map
PIE_SCALE = 0.60         # biggest pie in deg-area; raise/lower if pies are too small/crowd the map
LABEL_HUBS = True        # annotate each bus (hub) with its cluster name
EXTENT = [32.5, 48.5, 3.0, 15.5]   # Ethiopia bbox [lon_min,lon_max,lat_min,lat_max];
                                   # set None to auto-fit to the buses instead
SAVE = None              # e.g. "plots/2050_dry_HEG_geo_map.png"

n = load_net(RUNS[RUN]["path"])

# capacity pies per (bus, carrier): generators + storage units, minus the backstop
parts = [n.generators.groupby(["bus", "carrier"]).p_nom_opt.sum()]
if len(n.storage_units):
    parts.append(n.storage_units.groupby(["bus", "carrier"]).p_nom_opt.sum())
cap = pd.concat(parts)
cap = cap[cap > 1.0]
cap = cap[~cap.index.get_level_values("carrier").str.contains("load", case=False)]

carriers = list(cap.index.get_level_values("carrier").unique())
cmap = {c: COLORS.get(c, "#999999") for c in carriers}

bus_scale = PIE_SCALE / cap.groupby(level=0).sum().max()
LINE_SCALE = 4.0 / max(n.lines.s_nom_opt.max(), 1.0)     # widest line ~4 pt
exist_w = n.lines.s_nom * LINE_SCALE
new_w = (n.lines.s_nom_opt - n.lines.s_nom).clip(lower=0) * LINE_SCALE

acb = n.buses[n.buses.carrier == "AC"]
fig, ax = plt.subplots(figsize=(9, 9.5), subplot_kw={"projection": ccrs.PlateCarree()})

# --- basemap: land + internal geographic detail so hubs can be located --------
ax.add_feature(cfeature.LAND, facecolor="#f4f1ea", zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor="#dbeaf2", zorder=0)
ax.add_feature(cfeature.LAKES.with_scale("50m"), facecolor="#c6dbef",
               edgecolor="#9ec6e0", linewidth=0.3, zorder=1)   # Lake Tana, Rift lakes
ax.add_feature(cfeature.RIVERS.with_scale("50m"), edgecolor="#9ec6e0",
               linewidth=0.5, zorder=1)                         # Blue Nile, Omo, Awash
ax.add_feature(cfeature.STATES.with_scale("50m"), edgecolor="#cfc9bd",
               linewidth=0.4, zorder=1)                         # regional (admin-1) boundaries
ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.7,
               edgecolor="#777", zorder=2)
ax.coastlines("50m", linewidth=0.5)

def net_plot(**kw):
    """n.plot() across PyPSA versions: 0.30 uses color_geomap, newer uses geomap_color."""
    try:
        n.plot(ax=ax, geomap=True, color_geomap=False, **kw)
    except TypeError:
        n.plot(ax=ax, geomap=True, geomap_color=False, **kw)

# pass 1: existing lines (light grey) + capacity pies
net_plot(bus_sizes=cap * bus_scale, bus_colors=cmap,
         line_widths=exist_w, line_colors="#b5b5b5", link_widths=0.0)
# pass 2: overlay newly optimized line capacity (dark rose), no buses
net_plot(bus_sizes=0, line_widths=new_w, line_colors="#9c5c5c", link_widths=0.0)

# IMPORTANT: set the extent AFTER n.plot() -- n.plot auto-fits to the buses and
# would otherwise override EXTENT (this is why the country looked cropped before).
if EXTENT:
    ax.set_extent(EXTENT, crs=ccrs.PlateCarree())
else:
    ax.set_extent([acb.x.min() - 2, acb.x.max() + 2,
                   acb.y.min() - 2, acb.y.max() + 2], crs=ccrs.PlateCarree())

# lat/lon gridlines with labels, for georeferencing
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="#bbb", alpha=0.5, linestyle="--")
gl.top_labels = gl.right_labels = False
gl.xlabel_style = gl.ylabel_style = {"size": 7, "color": "#666"}

# --- hub labels: name each bus so you can see which cluster is where ----------
if LABEL_HUBS:
    for bus, row in acb.iterrows():
        ax.annotate(str(bus), xy=(row.x, row.y), xytext=(4, 4),
                    textcoords="offset points", fontsize=7, fontweight="bold",
                    color="#222", zorder=6,
                    bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.6))

tech_handles = [mpatches.Patch(color=cmap[c], label=c)
                for c in sorted(carriers, key=lambda c: -cap.xs(c, level=1).sum())]
leg1 = ax.legend(handles=tech_handles, loc="upper left", fontsize=8,
                 frameon=True, title="Technology")
ax.add_artist(leg1)

gw = 1e3
line_handles = [Line2D([0], [0], color="#b5b5b5", lw=v * gw * LINE_SCALE,
                       label=f"{v:g} GW existing") for v in (0.5, 1, 2)]
line_handles += [Line2D([0], [0], color="#9c5c5c", lw=v * gw * LINE_SCALE,
                        label=f"{v:g} GW new") for v in (0.5, 1, 2)]
leg2 = ax.legend(handles=line_handles, loc="lower left", fontsize=8,
                 frameon=True, title="HVAC line capacity")
ax.add_artist(leg2)

ax.annotate("pie area scales with installed capacity (largest bus = "
            f"{cap.groupby(level=0).sum().max()/1e3:,.0f} GW)",
            xy=(0.99, 0.02), xycoords="axes fraction", ha="right", fontsize=8,
            bbox=dict(boxstyle="round", fc="white", alpha=0.8))

ax.set_title(f"Power system — {RUN}", fontsize=13)
if SAVE:
    fig.savefig(SAVE, dpi=200, bbox_inches="tight")
    print("saved", SAVE)

## Notes
- **Export figures** for the thesis with `plt.savefig("plots/<name>.png", dpi=200)` after any plot cell.
- **Maps**: `python plot_country_map.py <solved.nc> plots/<name>.png "<title>"` (capacity pies per node, PyPSA-Earth atlas style).
- The `_cache` keeps networks in memory — restart the kernel if RAM gets tight after loading many runs.
- Remember: 168H runs are mechanics tests, not results (§3e). Check `len(n.snapshots)` if unsure: 52 = weekly, 2190 = 4H, 8760 = hourly.